# Métricas, árboles y bosques

*Encuentro virtual 3 (tercera parte) — Medir el error en millones de pesos, entender de dónde sale cada predicción y promediar árboles hasta que la predicción se sostenga*

---

**Técnicas de Análisis Estadístico de Modelos Supervisados**

Este cuaderno se genera automáticamente a partir del cuadernillo del sitio del curso. La versión web, con los gráficos interactivos y el formato completo, está en [https://wilsonsr.github.io/tecnicas-modelos-supervisados/03-regresion/arboles-y-bosques.html](https://wilsonsr.github.io/tecnicas-modelos-supervisados/03-regresion/arboles-y-bosques.html).

Ejecuta las celdas en orden, de principio a fin. Si te saltas alguna, las siguientes fallarán: es la misma disciplina que se exige en las actividades del curso.


In [ ]:
# Celda añadida automáticamente al generar este cuaderno.
# Descarga los datos del curso si no están disponibles, de modo que el cuaderno
# funcione igual en Google Colab que en el repositorio clonado. Si ya tienes el
# repositorio, no descarga nada.

import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/Wilsonsr/tecnicas-modelos-supervisados/main/"
ARCHIVOS = [
        "datos/crudos/vivienda_bogota.csv",
        "datos/crudos/ausentismo_laboral.csv",
        "datos/procesados/vivienda_modelado.csv",
]

if not os.path.exists("../datos/crudos/vivienda_bogota.csv"):
    # Sin repositorio: se crea la estructura y se descargan los datos.
    os.makedirs("curso/cuadernos", exist_ok=True)
    os.chdir("curso/cuadernos")
    for archivo in ARCHIVOS:
        destino = os.path.join("..", archivo)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        if not os.path.exists(destino):
            urllib.request.urlretrieve(BASE_URL + archivo, destino)
    print("Datos del curso descargados.")
else:
    print("Datos del curso encontrados en el repositorio.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, RepeatedKFold,
                                     cross_val_score)
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             root_mean_squared_error,
                             mean_absolute_percentage_error, r2_score)

SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})


In [ ]:
observado = np.array([3.0, 2.5, -2.0, 2.3, 4.0, 7.2, 8.1])
predicho  = np.array([3.1, 2.5, -2.5, 2.45, 3.8, 7.8, 7.9])

tabla = pd.DataFrame({
    "observado": observado,
    "predicho": predicho,
    "residuo": predicho - observado,
    "|residuo|": np.abs(predicho - observado),
    "residuo²": (predicho - observado) ** 2,
})
tabla.loc["suma"] = ["", "", "", tabla["|residuo|"].sum(), tabla["residuo²"].sum()]
tabla


Toda métrica de regresión sale de esas dos últimas columnas. La diferencia está
en **cómo se resume el residuo**: en valor absoluto o al cuadrado.

### Las cinco métricas

$$
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}\left|y_i - \hat{y}_i\right|
\qquad
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2
$$

$$
\text{RMSE} = \sqrt{\text{MSE}}
\qquad
\text{MAPE} = \frac{100\,\%}{n}\sum_{i=1}^{n}\frac{\left|y_i-\hat{y}_i\right|}{\left|y_i\right|}
\qquad
R^2 = 1 - \frac{\sum\left(y_i-\hat{y}_i\right)^2}{\sum\left(y_i-\bar{y}\right)^2}
$$


In [ ]:
n = len(observado)
residuo = predicho - observado

a_mano = {
    "MAE":  np.abs(residuo).sum() / n,
    "MSE":  (residuo ** 2).sum() / n,
    "RMSE": np.sqrt((residuo ** 2).sum() / n),
    "MAPE": 100 * (np.abs(residuo) / np.abs(observado)).sum() / n,
    "R²":   1 - (residuo ** 2).sum() / ((observado - observado.mean()) ** 2).sum(),
}
sklearn_ = {
    "MAE":  mean_absolute_error(observado, predicho),
    "MSE":  mean_squared_error(observado, predicho),
    "RMSE": root_mean_squared_error(observado, predicho),
    "MAPE": 100 * mean_absolute_percentage_error(observado, predicho),
    "R²":   r2_score(observado, predicho),
}
pd.DataFrame({"a mano": a_mano, "scikit-learn": sklearn_}).round(4)


> **IMPORTANTE**
> **Las unidades: la razón por la que el RMSE existe**
>
> Si `y` está en millones de pesos, entonces:
>
> - el **MAE** queda en millones de pesos → *«nos equivocamos típicamente en 237 millones»*;
> - el **MSE** queda en **millones de pesos al cuadrado** → no se puede decir en una reunión;
> - el **RMSE** devuelve la escala original al sacar la raíz → *«nos equivocamos en 348 millones»*.
>
> El MSE no es una métrica para reportar: es la que los algoritmos **optimizan**,
> porque es derivable y se descompone bien. El RMSE es la misma cantidad
> traducida al idioma del problema. En la Parte 3 veremos esto literalmente: el
> árbol minimiza sumas de cuadrados mientras lo construimos.


### MAE y RMSE no reaccionan igual

Aquí está la diferencia que decide cuál usar. Dejamos las seis primeras
predicciones intactas y empeoramos **una sola**: la quinta pasa de fallar por
0.2 a fallar por 3.0.


In [ ]:
predicho_malo = predicho.copy()
predicho_malo[4] = observado[4] + 3.0          # el único cambio

comparacion = pd.DataFrame({
    "original": [mean_absolute_error(observado, predicho),
                 root_mean_squared_error(observado, predicho)],
    "con un error grande": [mean_absolute_error(observado, predicho_malo),
                            root_mean_squared_error(observado, predicho_malo)],
}, index=["MAE", "RMSE"])
comparacion["se multiplicó por"] = (comparacion["con un error grande"]
                                    / comparacion["original"])
comparacion.round(3)


El MAE se multiplicó por **`{comparacion.loc['MAE','se multiplicó por']:.2f}`**
y el RMSE por **`{comparacion.loc['RMSE','se multiplicó por']:.2f}`**.
Un único caso catastrófico mueve mucho más al RMSE, porque elevar al cuadrado
le da peso desproporcionado a los residuos grandes.

De ahí la regla práctica:

| Si en tu problema… | Reporta | Porque |
|---|---|---|
| todos los errores cuestan proporcional a su tamaño | **MAE** | es el error típico, en unidades reales |
| un error grande cuesta mucho más que varios pequeños | **RMSE** | penaliza la cola de errores |
| hace falta comparar entre variables de escalas distintas | **MAPE** | es relativo, en porcentaje |
| hay que explicar si el modelo aporta algo | **R²** | compara contra predecir siempre el promedio |

*Qué métrica pide cada problema*
> **ADVERTENCIA**
> **Dos trampas del MAPE y del R²**
>
> **MAPE** divide por `y`. Si algún valor observado es cero o cercano a cero, el
> porcentaje explota y la métrica deja de significar nada. En estos siete números
> hay un `-2.0`, y el MAPE sale
> `{sklearn_['MAPE']:.1f}` % a pesar de que los residuos son pequeños.
>
> **R²** no es una calificación sobre 100. Es la comparación contra el modelo
> base: $R^2 = 0$ significa *«igual que predecir siempre el promedio»* y
> $R^2 < 0$ significa *«peor que el promedio»*, lo cual sí ocurre. Un $R^2$ de
> 0.75 en vivienda puede ser excelente y en física experimental, inaceptable.


---

**PRUEBA · Mover una predicción y ver qué métrica se entera**  ·  *10 min*

En la celda anterior, en lugar de empeorar una predicción, reparte el mismo error total entre todas: suma `0.4` al valor absoluto del residuo de **cada** observación.

1. El error total es casi el mismo que con el error único grande. ¿Qué pasa ahora con el MAE? ¿Y con el RMSE?
2. ¿Cuál de las dos situaciones preferiría un avaluador: fallar poco en todos los inmuebles o fallar muy poco en casi todos y muchísimo en uno?
3. Si el RMSE de un modelo es casi igual a su MAE, ¿qué te dice eso sobre la forma de sus errores?

La respuesta de la tercera pregunta es la herramienta de diagnóstico más barata que tienes: **RMSE ≫ MAE significa que hay un grupo de casos donde el modelo falla a lo grande**, y vale la pena ir a buscarlo.

---

# Parte 2 · Los datos

Trabajamos con los inmuebles de Bogotá del Cuadernillo 2. La variable objetivo
es el precio en **millones de pesos**, para que el MAE se pueda leer en voz alta.


In [ ]:
vivienda = pd.read_csv("../datos/procesados/vivienda_modelado.csv")
filas_originales = len(vivienda)

# El Cuadernillo 2 dejó la base con filas repetidas. Con duplicados entre
# entrenamiento y prueba, los modelos flexibles "aciertan" porque reconocen
# filas que ya vieron: el torneo de modelos documenta ese episodio completo.
vivienda = vivienda.drop_duplicates().reset_index(drop=True)
vivienda["precio_mm"] = vivienda.valor_venta / 1e6

NUMERICAS = ["area_m2", "n_cuartos", "n_banos", "n_garajes"]
CATEGORICAS = ["tipo_inmueble", "zona"]

X = vivienda[NUMERICAS + CATEGORICAS]
y = vivienda["precio_mm"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA)

print(f"{filas_originales:,} filas originales → {len(vivienda):,} sin duplicados")
print(f"entrenamiento: {len(X_train):,}   prueba: {len(X_test):,}")
print(f"precio: mediana {y.median():.0f} mm · rango {y.min():.0f}–{y.max():.0f} mm")


> **IMPORTANTE**
> **Quitar duplicados no es limpieza cosmética**
>
> De las `{filas_originales:,}` filas originales quedan
> `{len(vivienda):,}`: más del 80 % eran repeticiones del mismo
> anuncio. Si no se eliminan **antes** de partir, el mismo inmueble aparece en
> entrenamiento y en prueba, y el árbol sin podar llega a $R^2 \approx 0.98$
> sin haber aprendido nada generalizable.
>
> Es la quinta pregunta de la lista de fugas del Cuadernillo 2, y es la que más
> se pasa por alto. El [torneo de modelos](https://wilsonsr.github.io/tecnicas-modelos-supervisados/torneo-de-modelos.html) reconstruye el
> episodio con el marcador antes y después.


Los árboles no necesitan escalamiento, pero sí necesitan números: las dos
variables categóricas se codifican una sola vez, con el mismo
`ColumnTransformer` para todos los modelos, para que ninguno gane por tener
mejor preparación de datos.


In [ ]:
preparador = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERICAS),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first",
                          sparse_output=False), CATEGORICAS),
])

columnas = [c.split("__")[-1]
            for c in preparador.fit(X_train).get_feature_names_out()]
A_train = pd.DataFrame(preparador.transform(X_train), columns=columnas)
A_test = pd.DataFrame(preparador.transform(X_test), columns=columnas)

print(f"{A_train.shape[1]} columnas:", ", ".join(columnas))


### El modelo base: la vara de medir


In [ ]:
base = DummyRegressor(strategy="mean").fit(A_train, y_train)
pred_base = base.predict(A_test)

def evaluar(nombre, y_real, y_pred):
    return {"modelo": nombre,
            "MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": root_mean_squared_error(y_real, y_pred),
            "R²": r2_score(y_real, y_pred)}

marcador = [evaluar("Base (promedio)", y_test, pred_base)]
pd.DataFrame(marcador).round(3).set_index("modelo")


Predecir siempre el promedio se equivoca en
**`{marcador[0]['MAE']:.0f}` millones de pesos** por inmueble, y su
$R^2$ es 0 por construcción. Cualquier modelo que no mejore esa cifra no
merece existir.

# Parte 3 · El árbol: de dónde sale cada predicción

Un árbol de regresión parte los datos en grupos con cortes sucesivos y, dentro
de cada grupo, **predice el promedio**. Nada más. Empezamos con un árbol
diminuto —dos niveles, cuatro grupos— precisamente para poder revisarlo entero.


In [ ]:
arbol_chico = DecisionTreeRegressor(max_depth=2,
                                    random_state=SEMILLA).fit(A_train, y_train)
print(export_text(arbol_chico, feature_names=columnas, decimals=1))


In [ ]:
# Figura: Árbol de profundidad 2. En cada hoja, `value` es el promedio del precio de las viviendas de entrenamiento que caen allí.
fig, ax = plt.subplots(figsize=(10, 4.6))
plot_tree(arbol_chico, feature_names=columnas, filled=True, rounded=True,
          impurity=False, precision=0, fontsize=8, ax=ax)
plt.tight_layout()
plt.show()


Leídas en voz alta, las reglas son las de un avaluador novato pero sensato:

> Si el inmueble tiene menos de 122 m², vale unos 437 millones. Si tiene entre
> 122 y 177 m², unos 881. Entre 177 y 278, unos 1 376. Por encima de 278 m²,
> unos 2 265.

### Cada hoja es un promedio, literalmente

Esta es la propiedad que vuelve al árbol tan útil para entender las métricas.
Calculamos a mano el promedio del precio en cada grupo y lo comparamos con lo
que predice el árbol.


In [ ]:
hoja = arbol_chico.apply(A_train)                   # en qué hoja cae cada fila

resumen = (pd.DataFrame({"hoja": hoja, "precio": y_train.values})
           .groupby("hoja")
           .agg(n=("precio", "size"),
                promedio_a_mano=("precio", "mean"),
                mediana=("precio", "median"),
                desv_est=("precio", "std"),
                mae_interno=("precio", lambda s: (s - s.mean()).abs().mean())))
resumen["predice_el_arbol"] = [arbol_chico.tree_.value[h].ravel()[0]
                               for h in resumen.index]
resumen.round(1)


Las dos últimas columnas coinciden hasta el decimal. **El árbol no calcula nada
más misterioso que un promedio por grupo.**

Y ahora la consecuencia importante. Si la predicción de una hoja es el promedio
del grupo, entonces el error del modelo dentro de esa hoja es exactamente
**la dispersión que queda dentro del grupo**. Súmese ponderado por el tamaño de
cada hoja:


In [ ]:
mae_ponderado = (resumen.n * resumen.mae_interno).sum() / resumen.n.sum()
mae_entrenamiento = mean_absolute_error(y_train, arbol_chico.predict(A_train))

print(f"dispersión interna promedio de las hojas : {mae_ponderado:7.2f} mm")
print(f"MAE del árbol en entrenamiento           : {mae_entrenamiento:7.2f} mm")


> **SUGERENCIA**
> **La idea que conecta la Parte 1 con la Parte 3**
>
> El MAE de un árbol **no es una nota externa**: es la heterogeneidad que el
> modelo no logró separar. Mirando la tabla de hojas se ve dónde está el error:
> la hoja de inmuebles pequeños tiene una dispersión interna de
> `{resumen.mae_interno.iloc[0]:.0f}` millones, y la de inmuebles
> grandes, de `{resumen.mae_interno.iloc[-1]:.0f}`.
>
> Reducir el error del modelo y **partir mejor los grupos grandes y heterogéneos
> son la misma tarea**. Eso es, exactamente, lo que hace crecer el árbol.


### ¿Cómo elige el árbol dónde cortar?

Sin magia: prueba todos los cortes posibles de todas las variables y se queda
con el que deja los dos grupos resultantes **lo más homogéneos posible**. La
homogeneidad se mide con la suma de cuadrados dentro de cada grupo:

$$
\text{SCE}(c) = \sum_{x_i \le c}\left(y_i - \bar{y}_{\text{izq}}\right)^2
              + \sum_{x_i > c}\left(y_i - \bar{y}_{\text{der}}\right)^2
$$

Reproduzcámoslo a mano sobre el área, que es la variable que el árbol eligió
para la raíz.


In [ ]:
# Figura: Suma de cuadrados dentro de los grupos para cada corte posible del área. El mínimo es el corte que elige el algoritmo.
area = A_train.area_m2.values
precio = y_train.values

def suma_de_cuadrados(corte):
    izq, der = precio[area <= corte], precio[area > corte]
    if len(izq) < 5 or len(der) < 5:
        return np.nan
    return ((izq - izq.mean()) ** 2).sum() + ((der - der.mean()) ** 2).sum()

cortes = np.arange(60, 401, 2.5)
sce = np.array([suma_de_cuadrados(c) for c in cortes])
sce_sin_partir = ((precio - precio.mean()) ** 2).sum()

mejor_corte = cortes[np.nanargmin(sce)]
corte_sklearn = arbol_chico.tree_.threshold[0]

fig, ax = plt.subplots()
ax.plot(cortes, sce / 1e6, color="#17808C", lw=2)
ax.axhline(sce_sin_partir / 1e6, color="#8A8F98", ls=":", lw=1.4,
           label="sin partir (un solo grupo)")
ax.axvline(mejor_corte, color="#B4543A", ls="--", lw=1.6,
           label=f"mínimo a mano: {mejor_corte:.1f} m²")
ax.set_xlabel("corte en el área (m²)")
ax.set_ylabel("suma de cuadrados dentro de los grupos\n(millones de mm²)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"mejor corte calculado a mano : {mejor_corte:.1f} m²")
print(f"corte que eligió scikit-learn: {corte_sklearn:.1f} m²")
print(f"reduce la suma de cuadrados en {100 * (1 - np.nanmin(sce) / sce_sin_partir):.1f} %")


Los dos cortes coinciden. **La curva es el algoritmo completo**: el árbol
recorre esa curva para cada variable, se queda con el mínimo global, y repite
el procedimiento dentro de cada grupo nuevo hasta que se le dice que pare.

> **NOTA**
> **¿Y el índice de Gini y la entropía?**
>
> Son el criterio equivalente **para clasificación**, donde la variable objetivo
> es una categoría y «homogéneo» significa *de una sola clase*. En regresión el
> criterio es el de arriba: sumas de cuadrados, es decir, MSE.
>
> Por eso `DecisionTreeRegressor` usa `criterion="squared_error"` por defecto y
> `DecisionTreeClassifier` usa `criterion="gini"`. Gini y entropía se trabajan en
> el [Cuadernillo 4](https://wilsonsr.github.io/tecnicas-modelos-supervisados/04-clasificacion/cuadernillo-04.html); aquí solo
> estorbarían.


---

**INTERPRETA · Leer el árbol como un avaluador**  ·  *10 min*

Con la tabla de hojas y el dibujo del árbol a la vista:

1. El árbol de profundidad 2 usó **solo el área** en sus tres cortes, aunque tenía 13 columnas disponibles. ¿Qué significa eso sobre el resto de las variables?
2. En la hoja de inmuebles grandes, el promedio es 2 265 y la mediana 2 100. ¿Qué forma tiene la distribución allí, y qué le pasa a la predicción del árbol por usar el promedio y no la mediana?
3. Un cliente con un apartamento de 123 m² y otro con uno de 121 m² reciben avalúos que difieren en cientos de millones. ¿Es defendible? ¿Cómo lo explicarías en el comité?

La tercera pregunta señala la debilidad estructural del árbol: predice **por escalones**. Es la razón de fondo por la que conviene promediar muchos árboles, que es lo que haremos en la Parte 5.

---

# Parte 4 · El árbol crece demasiado

El árbol de profundidad 2 es claramente insuficiente: cuatro avalúos posibles
para toda Bogotá. La tentación es dejarlo crecer sin límite. Veamos qué pasa.


In [ ]:
filas = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, 15, None]:
    m = DecisionTreeRegressor(max_depth=d, random_state=SEMILLA).fit(A_train, y_train)
    filas.append({
        "profundidad": "sin límite" if d is None else d,
        "hojas": m.get_n_leaves(),
        "RMSE entrenamiento": root_mean_squared_error(y_train, m.predict(A_train)),
        "RMSE prueba": root_mean_squared_error(y_test, m.predict(A_test)),
        "MAE prueba": mean_absolute_error(y_test, m.predict(A_test)),
        "R² prueba": r2_score(y_test, m.predict(A_test)),
    })
profundidades = pd.DataFrame(filas).set_index("profundidad")
profundidades.round(2)


In [ ]:
# Figura: Al crecer el árbol, el error de entrenamiento baja siempre y el de prueba baja, toca fondo y vuelve a subir. La distancia entre las dos curvas es el sobreajuste.
niveles = list(range(1, 21))
err_train, err_test = [], []
for d in niveles:
    m = DecisionTreeRegressor(max_depth=d, random_state=SEMILLA).fit(A_train, y_train)
    err_train.append(root_mean_squared_error(y_train, m.predict(A_train)))
    err_test.append(root_mean_squared_error(y_test, m.predict(A_test)))

fig, ax = plt.subplots()
ax.plot(niveles, err_train, "o-", color="#1D4E7C", lw=2, ms=4, label="entrenamiento")
ax.plot(niveles, err_test, "o-", color="#B4543A", lw=2, ms=4, label="prueba")
ax.axvline(niveles[int(np.argmin(err_test))], color="#2E6B4F", ls="--", lw=1.4,
           label=f"mejor en prueba: {niveles[int(np.argmin(err_test))]}")
ax.set_xlabel("profundidad máxima del árbol")
ax.set_ylabel("RMSE (millones de pesos)")
ax.set_xticks(niveles[::2])
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
_sin_limite = profundidades.loc["sin límite"]
_mejor = profundidades.drop(index="sin límite")["RMSE prueba"].idxmin()
print(f"árbol sin límite  : {_sin_limite['hojas']:.0f} hojas, "
      f"RMSE entrenamiento {_sin_limite['RMSE entrenamiento']:.0f}, "
      f"RMSE prueba {_sin_limite['RMSE prueba']:.0f} mm")
print(f"mejor de la tabla : profundidad {_mejor}, "
      f"RMSE prueba {profundidades.loc[_mejor, 'RMSE prueba']:.0f} mm")


El árbol sin límite baja su error de entrenamiento casi a cero y **empeora** en
prueba. Es el ejemplo más limpio de varianza alta que da este curso: con hojas
de una o dos viviendas, el modelo memoriza anuncios en lugar de aprender el
mercado.

### Elegir la profundidad sin mirar la prueba

La curva anterior es didáctica pero **tramposa**: usamos el conjunto de prueba
para elegir un hiperparámetro, así que ese conjunto ya no es una evaluación
limpia. La decisión correcta se toma con validación cruzada dentro del
entrenamiento.


In [ ]:
cv = RepeatedKFold(n_splits=5, n_repeats=6, random_state=SEMILLA)

filas = []
for d in range(2, 11):
    puntajes = -cross_val_score(
        DecisionTreeRegressor(max_depth=d, random_state=SEMILLA),
        A_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    filas.append({"profundidad": d, "RMSE medio": puntajes.mean(),
                  "dispersión": puntajes.std()})
cv_prof = pd.DataFrame(filas).set_index("profundidad")
cv_prof.round(1)


In [ ]:
# Figura: RMSE promedio de validación cruzada ± una desviación estándar entre pliegues. Las profundidades cuyo intervalo se solapa con el del mínimo son empates técnicos.
d_min = int(cv_prof["RMSE medio"].idxmin())
umbral = cv_prof.loc[d_min, "RMSE medio"] + cv_prof.loc[d_min, "dispersión"]

# Entre las opciones MÁS SIMPLES que el mínimo, las que caen dentro de una
# desviación empatan con él. Se toma la más simple de todas.
candidatas = cv_prof.loc[:d_min]
empatan = candidatas.index[candidatas["RMSE medio"] <= umbral].tolist()
d_elegida = min(empatan)

fig, ax = plt.subplots()
ax.errorbar(cv_prof.index, cv_prof["RMSE medio"], yerr=cv_prof["dispersión"],
            fmt="o-", color="#1D4E7C", lw=1.8, ms=5, capsize=4)
ax.axhline(umbral, color="#8A8F98", ls=":", lw=1.4,
           label="mínimo + 1 desviación")
ax.axvline(d_elegida, color="#2E6B4F", ls="--", lw=1.6,
           label=f"elegida: {d_elegida}")
ax.set_xlabel("profundidad máxima")
ax.set_ylabel("RMSE de validación cruzada")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"mínimo en profundidad {d_min} ({cv_prof.loc[d_min,'RMSE medio']:.1f} mm)")
print(f"empatan dentro de una desviación: {empatan}")
print(f"se elige la más simple del empate: profundidad {d_elegida}")


> **SUGERENCIA**
> **La regla de la desviación estándar**
>
> El mínimo de una curva de validación cruzada casi nunca es significativo: las
> profundidades vecinas suelen caer dentro de la dispersión entre pliegues. Aquí
> `{len(empatan)}` profundidades empatan con el mínimo.
>
> Cuando varias opciones empatan, **se toma la más simple**. Un árbol con menos
> niveles tiene menos hojas, se explica en una diapositiva y es más estable
> frente a datos nuevos. Esto no es pereza estadística: es el mismo principio de
> parsimonia que justifica Ridge y Lasso en el Cuadernillo 3.


In [ ]:
arbol = DecisionTreeRegressor(max_depth=d_elegida,
                              random_state=SEMILLA).fit(A_train, y_train)
arbol_sin_podar = DecisionTreeRegressor(random_state=SEMILLA).fit(A_train, y_train)

marcador.append(evaluar(f"Árbol podado (prof. {d_elegida})",
                        y_test, arbol.predict(A_test)))
marcador.append(evaluar("Árbol sin podar", y_test,
                        arbol_sin_podar.predict(A_test)))
pd.DataFrame(marcador).round(3).set_index("modelo")


---

**COMPARA · Dos formas de controlar el crecimiento**  ·  *15 min*

`max_depth` no es la única manera de impedir que un árbol memorice. Repite la validación cruzada cambiando, uno a la vez:

1. `min_samples_leaf` (mínimo de viviendas por hoja): prueba 1, 5, 20, 50 dejando `max_depth=None`.
2. `ccp_alpha` (poda por costo-complejidad): prueba 0, 100, 1000, 5000.

Luego responde: ¿los tres caminos llegan a un error parecido? ¿Cuál de los tres le resulta más fácil de defender a un comité: «limité la profundidad a 4», «exigí al menos 20 viviendas por hoja» o «podé con alfa igual a 1 000»?

Pista: el que se puede traducir a una frase sobre el negocio, no sobre el algoritmo.

---

# Parte 5 · Un árbol es inestable; un bosque, menos

Un árbol podado ya es razonable, pero tiene un defecto que las métricas
globales esconden: **cambia mucho si cambian un poco los datos**. Para verlo,
entrenamos 100 árboles sobre 100 remuestreos con reemplazo del mismo
entrenamiento y observamos cuánto varía la predicción del *mismo* inmueble.


In [ ]:
generador = np.random.default_rng(SEMILLA)
predicciones = []
for _ in range(100):
    indices = generador.integers(0, len(A_train), len(A_train))
    arbolito = DecisionTreeRegressor(random_state=SEMILLA).fit(
        A_train.iloc[indices], y_train.iloc[indices])
    predicciones.append(arbolito.predict(A_test))
predicciones = np.array(predicciones)        # 100 árboles × n viviendas

dispersion = pd.DataFrame({
    "area_m2": X_test.area_m2.values,
    "precio_real": y_test.values,
    "desv_entre_arboles": predicciones.std(axis=0),
})
dispersion["tramo de área"] = pd.cut(dispersion.area_m2,
                                     [0, 80, 120, 180, 280, 600])
(dispersion.groupby("tramo de área", observed=True)
 .agg(viviendas=("area_m2", "size"),
      precio_promedio=("precio_real", "mean"),
      desacuerdo_entre_arboles=("desv_entre_arboles", "mean"))
 .round(1))


Los árboles están de acuerdo en los apartamentos pequeños y **se desacuerdan
profundamente en los grandes**, donde hay pocos datos: cientos de millones de
diferencia según qué remuestreo le tocó a cada árbol. Un solo árbol, en esa
zona, es casi un sorteo.


In [ ]:
# Figura: Predicción para el inmueble más inestable del conjunto de prueba, promediando cada vez más árboles. Lo que oscila con pocos árboles se estabiliza con unas decenas.
peor = int(dispersion.desv_entre_arboles.idxmax())
serie = predicciones[:, peor]
acumulado = np.cumsum(serie) / np.arange(1, len(serie) + 1)

fig, ax = plt.subplots()
ax.plot(range(1, len(serie) + 1), serie, "o", color="#C9D4DC", ms=4,
        label="un árbol individual")
ax.plot(range(1, len(serie) + 1), acumulado, color="#17808C", lw=2.2,
        label="promedio acumulado")
ax.axhline(y_test.iloc[peor], color="#B4543A", ls="--", lw=1.5,
           label=f"precio real: {y_test.iloc[peor]:.0f} mm")
ax.set_xlabel("número de árboles promediados")
ax.set_ylabel("predicción (millones de pesos)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"inmueble de {X_test.area_m2.iloc[peor]:.0f} m², precio real {y_test.iloc[peor]:.0f} mm")
print(f"un árbol solo  : entre {serie.min():.0f} y {serie.max():.0f} mm")
print(f"promedio de 100: {serie.mean():.0f} mm")


> **ADVERTENCIA**
> **Promediar quita varianza, no sesgo**
>
> Mírese el gráfico con cuidado: el promedio **se estabiliza**, pero se estabiliza
> lejos del precio real. En esa zona del mercado el bosque se equivoca de forma
> consistente, y promediar más árboles no lo corrige: todos los árboles comparten
> el mismo problema, que es la falta de datos de inmuebles grandes.
>
> Promediar reduce el componente de **varianza** del error —el que depende de qué
> muestra tocó— y no toca el de **sesgo** —el que viene del modelo y de los datos
> disponibles—. Es la descomposición del Cuadernillo 1, ahora en un caso concreto:
> si el error es por varianza, promediar ayuda; si es por sesgo, hace falta más
> información, no más árboles.


Eso es **bagging** (*bootstrap aggregating*): remuestrear, entrenar un árbol en
cada remuestreo y promediar. Cada árbol es ruidoso, pero sus ruidos no apuntan
al mismo lado y al promediarlos se cancelan.

`RandomForestRegressor` hace esto y **una cosa más**: en cada corte considera
solo un subconjunto aleatorio de las variables. Sin eso, todos los árboles
empezarían cortando por el área y se parecerían demasiado entre sí; al forzar
variedad, sus errores quedan menos correlacionados y el promedio gana más.


In [ ]:
bosque = RandomForestRegressor(n_estimators=500, random_state=SEMILLA,
                               n_jobs=-1).fit(A_train, y_train)
marcador.append(evaluar("Random Forest (500 árboles)", y_test,
                        bosque.predict(A_test)))
pd.DataFrame(marcador).round(3).set_index("modelo")


### ¿Cuántos árboles?


In [ ]:
# Figura: El error cae rápido con los primeros árboles y luego se estanca. Añadir más no deteriora el modelo: solo cuesta tiempo.
cantidades = [1, 3, 5, 10, 25, 50, 100, 200, 300, 500]
rmse_n = []
for n in cantidades:
    b = RandomForestRegressor(n_estimators=n, random_state=SEMILLA,
                              n_jobs=-1).fit(A_train, y_train)
    rmse_n.append(root_mean_squared_error(y_test, b.predict(A_test)))

fig, ax = plt.subplots()
ax.plot(cantidades, rmse_n, "o-", color="#2E6B4F", lw=2, ms=5)
ax.set_xscale("log")
ax.set_xticks(cantidades)
ax.set_xticklabels(cantidades)
ax.set_xlabel("número de árboles (escala logarítmica)")
ax.set_ylabel("RMSE de prueba (millones de pesos)")
plt.tight_layout()
plt.show()

print(f"  1 árbol : RMSE {rmse_n[0]:.0f} mm")
print(f" 25 árboles: RMSE {rmse_n[4]:.0f} mm")
print(f"500 árboles: RMSE {rmse_n[-1]:.0f} mm")


> **SUGERENCIA**
> **El único hiperparámetro que no hay que ajustar**
>
> `n_estimators` **no sobreajusta**. A diferencia de la profundidad, más árboles
> nunca empeoran el modelo de forma sistemática: la curva se aplana y lo único
> que crece es el tiempo de cómputo. Las subidas pequeñas que se ven en la curva
> son ruido de un solo conjunto de prueba, no deterioro.
>
> La regla práctica: tomar el número donde la curva se aplanó (aquí, cerca de
> 100) y subirlo si hay tiempo de sobra. Los hiperparámetros que **sí** hay que
> ajustar en un bosque son `max_features` y los que limitan cada árbol
> (`min_samples_leaf`, `max_depth`).


# Parte 6 · El marcador final

Agregamos la regresión lineal del Cuadernillo 3 para que la comparación tenga
sentido, y acompañamos cada modelo con su **dispersión en validación cruzada**:
sin ella no se puede afirmar que un modelo le gana a otro.


In [ ]:
competidores = {
    "Base (promedio)": DummyRegressor(strategy="mean"),
    "Regresión lineal": Pipeline([("escalar", StandardScaler()),
                                  ("modelo", LinearRegression())]),
    f"Árbol podado (prof. {d_elegida})": DecisionTreeRegressor(
        max_depth=d_elegida, random_state=SEMILLA),
    "Árbol sin podar": DecisionTreeRegressor(random_state=SEMILLA),
    "Random Forest": RandomForestRegressor(n_estimators=500,
                                           random_state=SEMILLA, n_jobs=-1),
}
cv_final = RepeatedKFold(n_splits=5, n_repeats=4, random_state=SEMILLA)

filas = []
for nombre, modelo in competidores.items():
    modelo.fit(A_train, y_train)
    pred = modelo.predict(A_test)
    puntajes = -cross_val_score(modelo, A_train, y_train, cv=cv_final,
                                scoring="neg_root_mean_squared_error")
    filas.append({**evaluar(nombre, y_test, pred),
                  "RMSE validación cruzada": puntajes.mean(),
                  "dispersión": puntajes.std()})

final = pd.DataFrame(filas).set_index("modelo").sort_values("MAE")
final.round(2)


In [ ]:
# Figura: MAE y RMSE de cada modelo. El orden del podio cambia según la métrica: el bosque gana en error típico, la regresión lineal en errores grandes.
orden = final.sort_values("MAE").index
pos = np.arange(len(orden))
fig, ax = plt.subplots(figsize=(7.4, 4.2))
ax.barh(pos - 0.2, final.loc[orden, "MAE"], height=0.38,
        color="#17808C", label="MAE")
ax.barh(pos + 0.2, final.loc[orden, "RMSE"], height=0.38,
        color="#1D4E7C", label="RMSE")
ax.set_yticks(pos)
ax.set_yticklabels(orden, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("millones de pesos")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()


El resultado es más interesante que un ganador limpio:


In [ ]:
lin, bos = final.loc["Regresión lineal"], final.loc["Random Forest"]
brecha = abs(bos["RMSE validación cruzada"] - lin["RMSE validación cruzada"])
dispersion_tipica = (bos["dispersión"] + lin["dispersión"]) / 2

print(f"MAE : bosque {bos['MAE']:.0f} vs lineal {lin['MAE']:.0f} mm"
      f"  → el bosque acierta mejor el inmueble típico")
print(f"RMSE: bosque {bos['RMSE']:.0f} vs lineal {lin['RMSE']:.0f} mm"
      f"  → la lineal controla mejor los errores grandes")
print(f"\nvalidación cruzada: diferencia de {brecha:.1f} mm "
      f"frente a una dispersión de {dispersion_tipica:.0f} mm")
print(f"la diferencia es {brecha / dispersion_tipica:.2f} veces la dispersión "
      f"→ empate técnico")


> **IMPORTANTE**
> **La conclusión que importa**
>
> El Random Forest **sí mejora** el error típico: el MAE baja de
> `{lin['MAE']:.0f}` a `{bos['MAE']:.0f}` millones, unos
> `{lin['MAE'] - bos['MAE']:.0f}` millones menos de error por
> inmueble. Eso es real y es relevante para un avalúo masivo.
>
> Pero su RMSE es ligeramente **peor**, y en validación cruzada los dos modelos
> están empatados dentro del ruido. Es decir: el bosque reparte mejor el error
> entre los inmuebles comunes, y falla un poco más fuerte en los raros.
>
> Dos lecciones, las dos de este curso:
>
> 1. **La métrica elige al ganador.** Quien reporte solo una de las dos está
>    decidiendo —tal vez sin saberlo— qué tipo de error le importa.
> 2. **Un algoritmo más complejo no gana automáticamente.** Con 1 500 registros,
>    seis variables y una relación dominada por el área, la regresión lineal
>    sigue siendo competitiva. La complejidad se justifica con evidencia, no con
>    el nombre del método.


---

**DECIDE · ¿Qué modelo le entregas al comité?**  ·  *15 min*

El comité va a usar el modelo para dos cosas distintas. Elige un modelo para cada una y escribe **dos frases** de justificación, citando la métrica que respalda tu decisión:

1. **Filtro masivo:** estimar de una vez el valor de 20 000 inmuebles para priorizar cuáles visita un avaluador humano.
2. **Peritaje individual:** sustentar ante un juzgado el valor de un inmueble de 420 m² en una zona con pocas transacciones.

Y una tercera pregunta, la más difícil: si el desempeño empata, **¿qué criterios no métricos deberían decidir?** Nombra al menos tres.

---

# Parte 7 · ¿Qué aprendió el bosque?

Un bosque de 500 árboles no se puede dibujar, pero sí se le puede preguntar qué
variables usó. Hay dos formas de hacerlo y **no dan la misma respuesta**.


In [ ]:
por_impureza = pd.Series(bosque.feature_importances_, index=columnas)

permutada = permutation_importance(
    bosque, A_test, y_test, n_repeats=20, random_state=SEMILLA,
    scoring="neg_root_mean_squared_error", n_jobs=-1)

importancias = pd.DataFrame({
    "por impureza (entrenamiento)": por_impureza,
    "por permutación (prueba, en mm de RMSE)": permutada.importances_mean,
}).sort_values("por impureza (entrenamiento)", ascending=False)
importancias.round(3)


In [ ]:
# Figura: Importancia por permutación: cuánto empeora el RMSE de prueba al desordenar cada variable. Una importancia negativa significa que el modelo predice mejor sin esa variable.
top = importancias.head(6).iloc[::-1]
fig, ax = plt.subplots()
colores = ["#B4543A" if v < 0 else "#2E6B4F"
           for v in top["por permutación (prueba, en mm de RMSE)"]]
ax.barh(top.index, top["por permutación (prueba, en mm de RMSE)"], color=colores)
ax.axvline(0, color="#4A5560", lw=1)
ax.set_xlabel("aumento del RMSE al permutar la variable (millones de pesos)")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()


> **ADVERTENCIA**
> **Por qué no basta con `feature_importances_`**
>
> La importancia por impureza se calcula **en entrenamiento** y mide cuánto
> ayudó cada variable a partir los nodos. Tiene dos sesgos conocidos: favorece a
> las variables **continuas y con muchos valores distintos** (tienen más cortes
> posibles para «acertar») y puede premiar a una variable que solo sirvió para
> memorizar.
>
> La importancia por permutación se calcula **en prueba**: desordena una columna
> y mide cuánto empeora la predicción. Responde la pregunta que interesa, que es
> *«¿cuánto perdería si dejara de medir esta variable?»*.
>
> Aquí las dos coinciden en que el área manda, pero discrepan en el resto del
> orden. Y `n_banos` recibe importancia positiva por impureza y **negativa** por
> permutación: el modelo predice igual o mejor sin ella.


In [ ]:
# Figura: Predicción del bosque frente al precio real en el conjunto de prueba. La nube se abre al subir el precio: el error no está repartido por igual.
pred_bosque = bosque.predict(A_test)
fig, ax = plt.subplots(figsize=(6.4, 6))
ax.scatter(y_test, pred_bosque, s=16, alpha=0.45, color="#17808C",
           edgecolor="none")
lim = [0, max(y_test.max(), pred_bosque.max()) * 1.03]
ax.plot(lim, lim, color="#B4543A", ls="--", lw=1.5, label="predicción perfecta")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("precio real (millones de pesos)")
ax.set_ylabel("precio predicho por el bosque")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
errores = pd.DataFrame({"precio_real": y_test.values,
                        "predicho": pred_bosque})
errores["error"] = errores.predicho - errores.precio_real
errores["quintil de precio"] = pd.qcut(errores.precio_real, 5)

(errores.groupby("quintil de precio", observed=True)
 .agg(viviendas=("error", "size"),
      sesgo_medio=("error", "mean"),
      MAE=("error", lambda s: s.abs().mean()))
 .round(1))


El error no está repartido: el MAE del quintil más barato es de decenas de
millones y el del más caro, de cientos. **El modelo se equivoca mucho más donde
los precios son dispersos y hay menos datos.**

El **sesgo medio** cuenta algo distinto y más sutil. Es positivo en los cuatro
quintiles bajos y negativo en el más alto: el bosque **sobreestima un poco los
inmuebles baratos y subestima los caros**. Es el encogimiento hacia el centro
que cabe esperar de un modelo que predice promedios de grupos: ninguna hoja
puede predecir por debajo del inmueble más barato de su grupo ni por encima del
más caro. En todos los tramos, eso sí, el sesgo es pequeño frente al MAE del
mismo tramo, así que el problema principal del bosque es la magnitud del error
en los inmuebles caros, no su dirección.

Esto explica el empate del marcador. El RMSE está dominado por ese último
quintil, donde la regresión lineal —que impone una tendencia en lugar de
promediar vecinos— se sostiene algo mejor. El MAE está dominado por los cuatro
primeros, donde el bosque es claramente superior.

# Parte 8 · Lo que el bosque no puede hacer

Antes de cerrar, una limitación que no aparece en ninguna métrica y que
sorprende a quien llega al bosque creyendo que reemplaza al modelo estadístico.


In [ ]:
# Figura: Predicción para un apartamento de 3 cuartos, 3 baños y 2 garajes en la zona norte, variando solo el área. Más allá del área máxima observada en entrenamiento, el bosque se queda plano.
lineal = Pipeline([("escalar", StandardScaler()),
                   ("modelo", LinearRegression())]).fit(A_train, y_train)

plantilla = pd.DataFrame(0.0, index=[0], columns=columnas)
plantilla.loc[0, ["n_cuartos", "n_banos", "n_garajes", "zona_NORTE"]] = [3, 3, 2, 1]

areas = np.arange(40, 1001, 10)
rejilla = pd.concat([plantilla] * len(areas), ignore_index=True)
rejilla["area_m2"] = areas

area_max = A_train.area_m2.max()
fig, ax = plt.subplots()
ax.plot(areas, lineal.predict(rejilla), color="#1D4E7C", lw=2.2,
        label="regresión lineal")
ax.plot(areas, bosque.predict(rejilla), color="#2E6B4F", lw=2.2,
        label="Random Forest")
ax.axvline(area_max, color="#B4543A", ls="--", lw=1.5,
           label=f"área máxima observada: {area_max:.0f} m²")
ax.set_xlabel("área (m²)")
ax.set_ylabel("precio predicho (millones de pesos)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

for a in [200, 400, 600, 1000]:
    f = plantilla.copy(); f["area_m2"] = a
    print(f"{a:5d} m² → lineal {lineal.predict(f)[0]:7.0f} mm"
          f" | bosque {bosque.predict(f)[0]:7.0f} mm")


Un árbol predice el promedio de una hoja, y ninguna hoja contiene inmuebles de
más de `{area_max:.0f}` m² porque no existen en los datos. Por eso
**el bosque es incapaz de extrapolar**: para 600 m² y para 1 000 m² devuelve
exactamente el mismo valor. La regresión lineal, en cambio, prolonga la
pendiente estimada —lo cual puede ser razonable o absurdo, pero al menos es una
afirmación revisable.

| | Regresión lineal | Árbol podado | Random Forest |
|---|---|---|---|
| **Interpretación** | coeficiente por variable, con intervalo | reglas legibles, se dibuja | caja negra; solo importancias |
| **Inferencia** | errores estándar, pruebas de hipótesis | no | no |
| **Extrapolación** | sí, prolonga la tendencia | no, se queda plano | no, se queda plano |
| **No linealidades e interacciones** | hay que especificarlas | las encuentra solo | las encuentra solo |
| **Supuestos** | linealidad, homocedasticidad, normalidad de residuos | casi ninguno | casi ninguno |
| **Estabilidad** | alta | baja | alta |
| **Costo de cómputo** | milisegundos | milisegundos | segundos a minutos |

*Para qué sirve cada uno*
> **NOTA**
> **Por qué este curso no abandona el modelo estadístico**
>
> El bosque gana donde hay que **predecir** mucho y rápido sobre casos parecidos
> a los observados. El modelo lineal gana donde hay que **explicar**, cuantificar
> la incertidumbre de un efecto o defender una cifra fuera del rango conocido
> —exactamente el terreno de un peritaje o de una política pública.
>
> breiman2001 llamó a esto las dos culturas del modelamiento, y shmueli2010 lo
> formuló como la distinción entre explicar y predecir. No es una jerarquía: es
> una elección que depende de la pregunta, y saber cuál hacer es el objetivo de
> este curso.


Lo que debes recordar

- **MAE** es el error típico en unidades reales; **RMSE** castiga los errores grandes; **MSE** no es reportable porque está en unidades al cuadrado; **MAPE** explota si hay valores cerca de cero; **R²** compara contra predecir el promedio.

- Si el RMSE es mucho mayor que el MAE, hay un grupo de casos donde el modelo falla a lo grande. Vale la pena buscarlo.

- La predicción de un árbol de regresión es **el promedio de su hoja**, y el error del modelo es la dispersión que queda dentro de las hojas. Reducir el error y partir mejor los grupos heterogéneos son la misma tarea.

- El árbol elige cada corte **minimizando la suma de cuadrados** dentro de los grupos resultantes. El criterio es MSE; Gini y entropía son el equivalente para clasificación.

- Un árbol sin podar memoriza: baja el error de entrenamiento casi a cero y empeora en prueba. La profundidad se elige con **validación cruzada**, nunca mirando el conjunto de prueba.

- Cuando varias configuraciones empatan dentro de la dispersión, **se toma la más simple**.

- Un solo árbol es inestable, y lo es más donde hay menos datos. **Random Forest promedia muchos árboles** entrenados en remuestreos y con variables sorteadas en cada corte; los ruidos individuales se cancelan.

- `n_estimators` no sobreajusta: más árboles solo cuestan tiempo. Lo que sí hay que ajustar es `max_features` y los límites de cada árbol.

- Reporta siempre la **dispersión** de la validación cruzada. Sin ella no se puede declarar un ganador, y aquí la lineal y el bosque empatan.

- Usa **importancia por permutación**, no solo `feature_importances_`: la de impureza favorece a las variables continuas y se calcula en entrenamiento.

- Los modelos de árboles **no extrapolan**. Fuera del rango observado devuelven un valor constante.

## Lo mismo en R


## Métricas y árbol

```r
library(tidymodels)
library(rpart.plot)

vivienda <- read_csv("datos/procesados/vivienda_modelado.csv") |>
  distinct() |>                                  # quitar duplicados
  mutate(precio_mm = valor_venta / 1e6)

set.seed(42)
division      <- initial_split(vivienda, prop = 0.75)
entrenamiento <- training(division)
prueba        <- testing(division)

receta <- recipe(precio_mm ~ area_m2 + n_cuartos + n_banos + n_garajes +
                   tipo_inmueble + zona, data = entrenamiento) |>
  step_impute_median(all_numeric_predictors()) |>
  step_dummy(all_nominal_predictors())

arbol <- decision_tree(tree_depth = tune()) |>
  set_engine("rpart") |> set_mode("regression")

# metric_set es el equivalente de elegir las métricas a reportar
metricas <- metric_set(mae, rmse, rsq)

ajuste <- workflow() |> add_recipe(receta) |> add_model(arbol) |>
  tune_grid(resamples = vfold_cv(entrenamiento, v = 5, repeats = 6),
            grid = tibble(tree_depth = 2:10),
            metrics = metricas)

# select_by_one_std_err() aplica la regla de la desviación estándar
mejor <- select_by_one_std_err(ajuste, metric = "rmse", tree_depth)
```

## Random Forest

```r
bosque <- rand_forest(trees = 500, mtry = tune(), min_n = tune()) |>
  set_engine("ranger", importance = "permutation") |>
  set_mode("regression")

flujo_bosque <- workflow() |> add_recipe(receta) |> add_model(bosque)

ajuste_bosque <- tune_grid(
  flujo_bosque,
  resamples = vfold_cv(entrenamiento, v = 5),
  grid = 20,
  metrics = metricas)

final <- finalize_workflow(flujo_bosque, select_best(ajuste_bosque, metric = "rmse")) |>
  last_fit(division, metrics = metricas)

collect_metrics(final)

# vip::vip() dibuja la importancia por permutación que pidió ranger
library(vip)
extract_fit_parsnip(final) |> vip()
```


En R, `select_by_one_std_err()` implementa directamente la regla de parsimonia
de la Parte 4, y `ranger` calcula la importancia por permutación si se le pide
en `set_engine()`. El equivalente de `permutation_importance()` de
`scikit-learn` sobre un modelo ya entrenado es `vip::vi_permute()`.

## Conexión con la Actividad 2

> **NOTA**
> **Actividad institucional 2 · Regresión lineal y regularizada (20 %, semanas 4 y 5)**
>
> La actividad evalúa **modelos lineales**: supuestos, interpretación de
> coeficientes y validación. Este cuadernillo no la reemplaza. Lo que aporta:
>
> - **El modelo base y la tabla de métricas** con el formato que se espera, y la
>   justificación explícita de por qué se reporta MAE o RMSE y no «la que salga».
> - **La dispersión de la validación cruzada** junto a cada promedio. Una tabla
>   de métricas sin dispersión se devuelve para corrección.
> - **La eliminación de duplicados antes de partir**, que a partir de esta semana
>   se espera en todas las entregas.
> - Si incluyes un Random Forest como comparación, hazlo con la misma partición,
>   la misma semilla y el mismo preprocesamiento, y **reporta también sus
>   limitaciones**: un bosque presentado como superior sin evidencia resta en
>   lugar de sumar.


El [Cuadernillo 4](https://wilsonsr.github.io/tecnicas-modelos-supervisados/04-clasificacion/cuadernillo-04.html) retoma estos mismos
árboles con la variable objetivo convertida en categoría. Allí cambia el
criterio de corte (Gini y entropía), cambia el marcador por completo
(sensibilidad, precisión, F1, AUC) y aparece una decisión que en regresión no
existe: **dónde poner el umbral**.

## Recursos adicionales

- james2023, capítulo 8 — árboles, bagging y bosques aleatorios, con la
  notación de este cuadernillo.
- breiman2001rf — el artículo original de Random Forest.
- hastie2009, sección 9.2 — el criterio de corte por sumas de cuadrados,
  desarrollado formalmente.
- breiman2001 y shmueli2010 — las dos culturas del modelamiento y la
  distinción entre explicar y predecir.
- [scikit-learn · Decision Trees](https://scikit-learn.org/stable/modules/tree.html) —
  documentación y advertencias prácticas.
- [scikit-learn · Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html) —
  incluye el detalle de los sesgos de la importancia por impureza.
